# 01.6 损失函数与优化器（Losses and Optimizers）

这一节开始回答两个训练中的核心问题：  

1. 模型到底“错了多少”
2. 参数应该往哪个方向更新

这两个问题分别由：  

- 损失函数（loss function）
- 优化器（optimizer）

## 学习目标

学完后你应该能

1. 理解损失函数在训练中的作用
2. 区分回归和分类常见损失
3. 理解 `MSELoss` 和 `CrossEntropyLoss` 的输入输出要求
4. 理解优化器在更新参数中的作用
5. 看懂 `zero_grad()`、`backward()`、`step()` 这三个动作
6. 为完整训练循环做准备

In [ ]:
import torch
import torch.nn as nn

## 1. 什么是损失函数

loss function 的作用是把“预测和真实值之间的差异”变成一个可以优化的数值。  

一般来说

- smaller loss -> 模型越接近目标
- larger loss -> 模型偏差越大

## 2. 回归中的 `MSELoss`

mean squared error 是回归任务中非常常见的损失。  

公式直觉

- 先算误差（compute prediction error）
- 再平方（square it）
- 最后求平均（then average）

In [ ]:
pred = torch.tensor([[2.5], [0.0], [2.0], [8.0]], dtype=torch.float32)
target = torch.tensor([[3.0], [-0.5], [2.0], [7.0]], dtype=torch.float32)

mse_loss = nn.MSELoss()
loss = mse_loss(pred, target)
print("MSE loss =", loss.item())

回归里，一个常见习惯是让 `pred` 和 `target` 的 shape 尽量一致。  


In [ ]:
# 练习 1
# 用 MSELoss 计算下面的 loss。
# Use MSELoss to compute the loss below.

pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([[1.5], [2.5], [2.0]])

# loss_fn =
# loss =
# print(loss.item())

In [ ]:
# 练习 1 参考答案

pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([[1.5], [2.5], [2.0]])
loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)
print(loss.item())

## 3. 分类中的 `CrossEntropyLoss`

多分类任务里，`CrossEntropyLoss` 非常常见。  

它的输入要求特别重要

- model output: 原始分数
- labels: 类别编号

一个常见误区

- 不要先手动做 `softmax` 再传给 `CrossEntropyLoss`  
  Do not manually apply `softmax` before passing outputs to `CrossEntropyLoss`.

In [ ]:
logits = torch.tensor(
    [
        [2.0, 0.5, -1.0],
        [0.1, 0.2, 2.5],
        [1.5, 1.1, 0.3],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 2, 1], dtype=torch.long)

ce_loss = nn.CrossEntropyLoss()
loss = ce_loss(logits, targets)

print("logits.shape =", logits.shape)
print("targets.shape =", targets.shape)
print("CrossEntropy loss =", loss.item())

形状要求

- `logits.shape == (batch_size, num_classes)`
- `targets.shape == (batch_size,)`

标签不是 one-hot，而是类别编号。  


In [ ]:
# 练习 2
# 用 CrossEntropyLoss 计算下面的分类损失。
# Use CrossEntropyLoss to compute the classification loss below.

logits = torch.tensor(
    [
        [1.0, 0.5],
        [0.2, 2.0],
        [1.2, 1.1],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 1, 0], dtype=torch.long)

# loss_fn =
# loss =
# print(loss.item())

In [ ]:
# 练习 2 参考答案

logits = torch.tensor(
    [
        [1.0, 0.5],
        [0.2, 2.0],
        [1.2, 1.1],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 1, 0], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)
print(loss.item())

## 4. 优化器

损失函数告诉你“错了多少”，优化器告诉你“参数该怎么改”。  

常见优化器

- `SGD`
- `Adam`

你现在先不必深入公式，但要知道它们都在根据梯度更新参数。  


In [ ]:
model = nn.Linear(2, 1)
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.01)

print("SGD optimizer / SGD 优化器:")
print(optimizer_sgd)
print()
print("Adam optimizer / Adam 优化器:")
print(optimizer_adam)

## 5. `zero_grad()`、`backward()`、`step()`
## `zero_grad()`, `backward()`, and `step()`

训练里最核心的三步动作就是：  

1. `zero_grad()`：清掉旧梯度
2. `backward()`：根据 loss 计算新梯度
3. `step()`：根据梯度更新参数

In [ ]:
torch.manual_seed(0)

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])

print("更新前参数 / parameters before update:")
for name, param in model.named_parameters():
    print(name, param.data)

optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()

print()
print("loss =", loss.item())
print()
print("更新后参数 / parameters after update:")
for name, param in model.named_parameters():
    print(name, param.data)

这一小段代码其实已经非常接近完整训练循环。  


In [ ]:
# 练习 3
# 用下面的模型和数据，做一次参数更新。
# Use the model and data below to perform one parameter update.
#
# 需要补全
# 1. optimizer.zero_grad()
# 2. pred = model(x)
# 3. loss = loss_fn(pred, y)
# 4. loss.backward()
# 5. optimizer.step()

torch.manual_seed(1)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])

# TODO

In [ ]:
# 练习 3 参考答案

torch.manual_seed(1)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])

optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()

print("loss =", loss.item())
for name, param in model.named_parameters():
    print(name, param.data)

## 6. `SGD` 和 `Adam` 的直觉区别

先记一个足够实用的直觉版本：  

- `SGD`：更基础、更直接（more basic and direct）
- `Adam`：通常更省心，默认设置下经常更快收敛

这不是绝对规则，但对初学阶段很有帮助。  


## 7. 任务类型、输出层、损失函数的对应关系

这是非常重要的一张脑内对照表：  

- 回归（regression:）
  输出维度常为 1
  常见损失 `MSELoss`
- 多分类（multiclass classification:）
  输出维度通常等于类别数
  常见损失 `CrossEntropyLoss`

如果这张对照表混乱，训练常常一开始就报 shape 或 dtype 错。  


In [ ]:
# 练习 4
# 判断下面场景更适合什么损失函数。
# Decide which loss function fits each scenario.
#
# 1. 预测明天温度
# 2. 把手写数字分成 10 类
#
# 请用一句话回答，并说明理由。
# Answer in one sentence and explain why.

参考回答

- temperature prediction: 更适合 `MSELoss`，因为目标是连续值（better suited to `MSELoss` because the target is continuous）
- 10-class classification: 更适合 `CrossEntropyLoss`，因为目标是离散类别（better suited to `CrossEntropyLoss` because the target is a discrete class）

## 8. 小结

本节的核心链条是：  

- 模型输出（model outputs）
- 损失函数把输出和目标比较
- 反向传播产生梯度（backpropagation produces gradients）
- 优化器根据梯度更新参数

你现在应该能回答

1. `MSELoss` 和 `CrossEntropyLoss` 各适合什么任务？
2. 为什么 `CrossEntropyLoss` 的标签通常是类别编号而不是 one-hot？
3. `zero_grad()`、`backward()`、`step()` 分别做什么？
4. 为什么任务类型会影响输出层设计和损失函数选择？

下一步建议

- 接下来最自然的是进入完整训练循环 notebook